# GNAS

Cohort: `GNAS`, Phenopacket Store v0.1.27 (n = 103).

- Pseudohypoparathyroidism Ia (PHP-Ia), [OMIM:103580](https://omim.org/entry/103580), n = 80
- Pseudopseudohypoparathyroidism (PPHP), [OMIM:612463](https://omim.org/entry/612463), n = 23

Both subtypes share the Albright hereditary osteodystrophy (AHO) features and are
separated by parent of origin: maternally inherited inactivating GNAS variants cause
PHP-Ia with multi-hormone resistance, paternally inherited variants cause PPHP with AHO
alone. Disease membership is therefore a clinically meaningful synergy target.

All individuals derive from a single published series (PMID 25802881, Thiele et al.
2015, Mol Genet Genomic Med 3:111-120).

In [1]:
from pathlib import Path

import pandas as pd
from ppktstore.registry import configure_phenopacket_registry

from phenosign import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
    SynergyAnalyzer,
    has_disease,
)

OUTDIR = Path("../case_studies/GNAS")
OUTDIR.mkdir(parents=True, exist_ok=True)

In [2]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store(release="0.1.27") as ps:
    phenopackets = list(ps.iter_cohort_phenopackets("GNAS"))

print(f"Loaded {len(phenopackets)} phenopackets")

Loaded 103 phenopackets


In [3]:
# has_disease works on the phenopacket directly, no GPSEA cohort needed here
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

In [4]:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
diseases_df

,label,n_individuals
disease_id,,
OMIM:103580,Pseudohypoparathyroidism Ia,80 (77.7%)
OMIM:612463,Pseudopseudohypoparathyroidism,23 (22.3%)


## Pairwise correlation

In [5]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
results = analyzer.compute_correlation_matrix()

t = results.results_table
print(f"tested pairs: {len(t)} | significant: {int((t['adj_p_value'] < 0.05).sum())}")
t.head()

Calculating pairwise correlation: 100%|██████████| 63/63 [00:01<00:00, 48.49it/s]

tested pairs: 63 | significant: 5


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,n_pmids,pmids
40,HP:0002905,Hyperphosphatemia,HP:0004363,Abnormal circulating calcium concentration,0.456887,0.000169,0.010672,26,4,17,24,71,1,25802881
59,HP:0004363,Abnormal circulating calcium concentration,HP:0100529,Abnormal circulating phosphate ion concentration,0.426352,0.000778,0.016333,22,17,4,24,67,1,25802881
30,HP:0002901,Hypocalcemia,HP:0002905,Hyperphosphatemia,0.409337,0.000689,0.016333,27,21,3,20,71,1,25802881
60,HP:0010049,Short metacarpal,HP:0034282,Subcutaneous ossification,-0.337860,0.001093,0.017218,8,15,58,21,102,1,25802881
36,HP:0002901,Hypocalcemia,HP:0100529,Abnormal circulating phosphate ion concentration,0.382227,0.001749,0.022044,23,21,3,20,67,1,25802881


In [6]:
results.save_correlation_results(
    output_file=str(OUTDIR / "correlation_results_GNAS.csv")
)

In [7]:
sig = results.results_table.query("adj_p_value < 0.05")
sig = sig.reindex(sig["correlation"].abs().sort_values(ascending=False).index)
sig[["HPO_A_label", "HPO_B_label", "correlation", "adj_p_value", "n_individuals", "n_pmids"]]

,HPO_A_label,HPO_B_label,correlation,adj_p_value,n_individuals,n_pmids
40,Hyperphosphatemia,Abnormal circulating calcium concentration,0.456887,0.010672,71,1
59,Abnormal circulating calcium concentration,Abnormal circulating phosphate ion concentration,0.426352,0.016333,67,1
30,Hypocalcemia,Hyperphosphatemia,0.409337,0.016333,71,1
36,Hypocalcemia,Abnormal circulating phosphate ion concentration,0.382227,0.022044,67,1
60,Short metacarpal,Subcutaneous ossification,-0.337860,0.017218,102,1


In [8]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort GNAS",
)

## Synergy: PHP-Ia vs PPHP

In [9]:
condition = dataset.get_condition(
    has_disease("OMIM:103580"),
    name="PHP-Ia vs PPHP",
)

condition.value_counts(dropna=False)

PHP-Ia vs PPHP
1.0    80
0.0    23
Name: count, dtype: int64

In [10]:
# lowered from the default of 30, the cohort splits 80/23
synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    min_individuals_for_synergy_calculation=25,
)
synergy_results = synergy_analyzer.compute_synergy_matrix(condition=condition)

st = synergy_results.results_table
print(
    f"rows={len(st)} | synergy {st['synergy'].min():.4f} to {st['synergy'].max():.4f}"
    f" | positive={(st['synergy'] > 0).sum()} negative={(st['synergy'] < 0).sum()}"
)
st.head()

Calculating pairwise synergy: 100%|██████████| 63/63 [00:00<00:00, 88.32it/s]


rows=52 | synergy -0.0570 to 0.0354 | positive=21 negative=31


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,synergy,p_value,adj_p_value,n(A:E/B:E)_y0,n(A:E/B:O)_y0,n(A:O/B:E)_y0,n(A:O/B:O)_y0,N_y0,n(A:E/B:E)_y1,n(A:E/B:O)_y1,n(A:O/B:E)_y1,n(A:O/B:O)_y1,N_y1,n_individuals,n_pmids,pmids
41,HP:0003165,Elevated circulating parathyroid hormone level,HP:0100529,Abnormal circulating phosphate ion concentration,0.007816,0.117764,0.498204,0,2,0,0,2,3,2,17,30,52,54,1,25802881
49,HP:0010049,Short metacarpal,HP:0034282,Subcutaneous ossification,0.027604,0.036963,0.498204,3,0,14,5,22,5,15,44,16,80,102,1,25802881
21,HP:0002901,Hypocalcemia,HP:0003165,Elevated circulating parathyroid hormone level,-0.020611,0.099800,0.498204,2,0,0,0,2,5,32,0,19,56,58,1,25802881
17,HP:0001513,Obesity,HP:0034282,Subcutaneous ossification,0.020482,0.085828,0.498204,12,5,5,0,22,12,14,37,17,80,102,1,25802881
4,HP:0001249,Intellectual disability,HP:0003165,Elevated circulating parathyroid hormone level,-0.052634,0.065868,0.498204,14,0,4,0,18,4,23,4,29,60,78,1,25802881


In [11]:
# save_synergy_results() drops everything below synergy >= 0.01,
# so keep the unfiltered table for the dissynergy side
synergy_results.results_table.to_csv(
    OUTDIR / "synergy_full_GNAS_OMIM_103580.csv", index=False
)
synergy_results.save_synergy_results(
    output_file=str(OUTDIR / "synergy_matrix_GNAS_OMIM_103580.csv")
)

In [12]:
synergy_results.plot_synergy_heatmap()

ValueError: No sufficient synergy pairs remain after filtering. Try adjusting `synergy_threshold` or `adj_pval_threshold`.

In [ ]:
COLS = ["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals", "n_pmids"]

print("--- top synergy ---")
display(st.nlargest(10, "synergy")[COLS])
print("--- top dissynergy ---")
display(st.nsmallest(10, "synergy")[COLS])